# Fase 07 — Cribado de elegibilidad a texto completo

Esta fase revisa las decisiones de título y resumen utilizando los textos completos disponibles. La decisión final se registra por fuente y conserva una base breve, localizadores de página, estado de validación humana y razón de exclusión cuando corresponda.

Principios:

- una inclusión por título y resumen sigue siendo provisional;
- la falta de acceso o texto ilegible no constituye una exclusión de elegibilidad;
- los paquetes de fragmentos sirven para priorizar y asistir la revisión, pero no sustituyen la lectura del documento completo;
- las decisiones asistidas por modelos permanecen como `not_reviewed` hasta validación humana.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from evidence_review.full_text_screening import (
    build_evidence_extraction_corpus,
    build_full_text_exclusion_log,
    completed_full_text_decisions,
    export_full_text_screening_prompts_jsonl,
    initialise_full_text_screening_sheet,
    load_full_text_screening_config,
    merge_full_text_screening_decisions,
    read_csv_robust,
    read_full_text_screening_decisions_jsonl,
    screening_summary,
    select_full_text_screening_packets,
    split_full_text_screening_outputs,
    validate_full_text_screening_sheet,
)

CONFIG_PATH = ROOT / "config" / "full_text_screening.yml"
config = load_full_text_screening_config(CONFIG_PATH)

INTERIM = ROOT / "data" / "interim"
INTERIM.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Configuration: {CONFIG_PATH.relative_to(ROOT)}")


## 1. Cargar el corpus, los fragmentos y los metadatos

La entrada principal procede de la fase 06. El archivo de metadatos de recuperación es opcional y añade tipo de fuente, DOI y URL cuando está disponible.


In [ ]:
paths = config["paths"]

corpus_path = ROOT / paths["corpus_csv"]
chunks_path = ROOT / paths["chunks_csv"]
metadata_path = ROOT / paths["retrieval_metadata_csv"]
working_path = ROOT / paths["working_csv"]

corpus, corpus_encoding = read_csv_robust(corpus_path)
chunks, chunks_encoding = read_csv_robust(chunks_path)

metadata = pd.DataFrame()
metadata_encoding = "not_loaded"
if metadata_path.exists():
    metadata, metadata_encoding = read_csv_robust(metadata_path)

existing = pd.DataFrame()
if working_path.exists():
    existing, _ = read_csv_robust(working_path)

print(f"Full-text corpus: {len(corpus)} [{corpus_encoding}]")
print(f"Document chunks: {len(chunks)} [{chunks_encoding}]")
print(f"Retrieval metadata: {len(metadata)} [{metadata_encoding}]")
print(f"Existing review rows: {len(existing)}")


## 2. Inicializar la hoja de revisión

La hoja conserva cualquier decisión previa por `source_id`. No incorpora el texto completo dentro del CSV de trabajo; el contenido permanece en el corpus y en los fragmentos.


In [ ]:
working = initialise_full_text_screening_sheet(
    corpus,
    metadata=metadata,
    existing=existing,
)

working.to_csv(working_path, index=False, encoding="utf-8-sig")

print(f"Review queue: {len(working)}")
print(f"Unique source_id: {working['source_id'].nunique()}")
print(f"Saved: {working_path.relative_to(ROOT)}")

display(screening_summary(working))
display(
    working[
        [
            "source_id",
            "title",
            "title_abstract_decision",
            "full_text_screening_required",
            "page_count",
            "parsing_status",
            "decision",
            "human_validation_status",
        ]
    ]
    .sort_values(
        ["full_text_screening_required", "title_abstract_decision"],
        ascending=[False, False],
    )
    .head(25)
)


## 3. Construir paquetes representativos y prompts

Los paquetes seleccionan fragmentos del inicio, final, cobertura interna y fragmentos con términos diagnósticos. Cada fragmento conserva páginas y `chunk_id`.

La celda genera prompts JSONL, pero no llama a ningún modelo externo.


In [ ]:
packets = select_full_text_screening_packets(chunks, config)

packet_path = ROOT / paths["packet_csv"]
prompts_path = ROOT / paths["prompts_jsonl"]

packets.to_csv(packet_path, index=False, encoding="utf-8-sig")
export_full_text_screening_prompts_jsonl(
    working,
    packets,
    config,
    prompts_path,
)

packet_summary = (
    packets.groupby("source_id", as_index=False)
    .agg(
        packet_chunks=("chunk_id", "count"),
        packet_characters=("character_count", "sum"),
        start_page=("start_page", "min"),
        end_page=("end_page", "max"),
    )
)

print(f"Packet rows: {len(packets)}")
print(f"Sources with packets: {packets['source_id'].nunique()}")
print(f"Saved: {packet_path.relative_to(ROOT)}")
print(f"Saved: {prompts_path.relative_to(ROOT)}")
display(packet_summary.head(25))


## 4. Importar decisiones asistidas opcionales

Formato esperado: un objeto JSON por línea con `source_id`, decisión, cuatro criterios, base de decisión y páginas. La importación está desactivada por defecto.

Los registros `accepted`, `corrected` o `rejected` no se sobrescriben salvo que se active explícitamente `OVERWRITE_HUMAN_VALIDATED`.


In [ ]:
IMPORT_MODEL_DECISIONS = False
OVERWRITE_HUMAN_VALIDATED = False

model_decisions_path = ROOT / paths["model_decisions_jsonl"]

if IMPORT_MODEL_DECISIONS:
    if not model_decisions_path.exists():
        raise FileNotFoundError(model_decisions_path)

    model_decisions = read_full_text_screening_decisions_jsonl(
        model_decisions_path
    )

    working = merge_full_text_screening_decisions(
        working,
        model_decisions,
        overwrite_human_validated=OVERWRITE_HUMAN_VALIDATED,
    )

    working.to_csv(working_path, index=False, encoding="utf-8-sig")
    print(f"Imported decisions: {len(model_decisions)}")
else:
    print(
        "Model-decision import disabled. "
        "Set IMPORT_MODEL_DECISIONS = True after reviewing the JSONL file."
    )

display(screening_summary(working))


## 5. Validar las decisiones

Las decisiones vacías son trabajo pendiente y no generan errores. Toda decisión completada debe respetar la lógica de criterios, usar vocabularios permitidos y contener una base y páginas verificables.


In [ ]:
working, working_encoding = read_csv_robust(working_path)

issues = validate_full_text_screening_sheet(working, config)
issues_path = ROOT / paths["issues_csv"]
issues.to_csv(issues_path, index=False, encoding="utf-8-sig")

print(f"Working encoding: {working_encoding}")
print(f"Validation issues: {len(issues)}")
display(issues.head(50))
display(screening_summary(working))


## 6. Exportar decisiones, exclusiones y corpus de la fase 08

Solo las fuentes con decisión `include` entran al corpus de extracción estructurada. Las decisiones `uncertain` y vacías permanecen fuera hasta resolverse.


In [ ]:
groups = split_full_text_screening_outputs(working)
decisions = completed_full_text_decisions(working)
exclusion_log = build_full_text_exclusion_log(working)
evidence_corpus = build_evidence_extraction_corpus(chunks, working)
flow = screening_summary(working)

output_frames = {
    paths["decisions_csv"]: decisions,
    paths["included_csv"]: groups["included"],
    paths["excluded_csv"]: groups["excluded"],
    paths["uncertain_csv"]: groups["uncertain"],
    paths["pending_csv"]: groups["pending"],
    paths["exclusion_log_csv"]: exclusion_log,
    paths["flow_summary_csv"]: flow,
    paths["evidence_extraction_corpus_csv"]: evidence_corpus,
}

for relative_path, frame in output_frames.items():
    path = ROOT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"{path.relative_to(ROOT)}: {len(frame)}")

display(flow)


## Criterio para avanzar a la fase 08

La extracción estructurada de hallazgos puede comenzar cuando:

1. `Validation issues = 0`;
2. cada fuente con texto completo tiene una decisión o una justificación explícita de incertidumbre;
3. las tres decisiones previamente `uncertain` han sido revisadas;
4. toda inclusión tiene base y páginas;
5. toda exclusión tiene una razón válida y páginas;
6. las decisiones asistidas utilizadas en el análisis han sido aceptadas o corregidas por una persona.

La siguiente fase será:

```text
notebooks/08_extract_structured_findings.ipynb
```
